# ViGoEmotions — PhoBERT learning-rate tuning
Bật **Runtime → Change runtime type → T4 GPU**. Tạo gói local bằng `python scripts/package_colab.py --include-results`, upload ZIP ở cell dưới rồi upload riêng CSV được cấp quyền.

Chạy lại cả `1e-5`, `2e-5`, `3e-5` trong cùng runtime; seed/split/batch/epoch cố định. Chọn bằng validation tại 0,5. Chưa có kết quả sweep tại thời điểm tạo notebook. Checkpoint tải riêng ở cell cuối; gói kết quả công khai không chứa dữ liệu nguồn hay trọng số.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()  # Chọn vigoemotions_colab.zip
archive = next(name for name in uploaded if name.endswith('.zip'))
ROOT = Path('/content/ViGoEmotions')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = (ROOT/item.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Unsafe ZIP path')
    z.extractall(ROOT)
os.chdir(ROOT)

In [ ]:
%pip install -q -r requirements-colab.txt
!apt-get -qq update
!apt-get -qq install -y default-jre-headless fonts-dejavu-core

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi chạy tiếp.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['java','-version'], check=True)

## Nạp dữ liệu được cấp phép
Chọn đủ `train.csv`, `val.csv`, `test.csv`. Dữ liệu nằm trong runtime riêng và không đưa vào gói kết quả tải xuống.

In [ ]:
uploaded = files.upload()
Path('data').mkdir(exist_ok=True)
for name in ('train.csv','val.csv','test.csv'):
    if name not in uploaded:
        raise FileNotFoundError(f'Thiếu {name}')
    Path('data',name).write_bytes(uploaded[name])
del uploaded

## Kiểm tra gói code và baseline
Cần ZIP có `--include-results` để report giữ được đủ baseline và EDA. Nếu đã chạy sweep trước đó, dùng runtime/gói mới với thư mục sweep chưa có kết quả.

In [ ]:
import sys, subprocess, json
sys.path.insert(0, str(ROOT/'scripts'))
for name in ('tfidf_lr','gru','phobert'):
    assert Path(f'reports/tables/{name}_metrics.csv').exists(), 'Tạo lại ZIP với --include-results'
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)
import torch
assert torch.cuda.is_available(), 'Hãy bật GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))

## Chạy ba learning rate
Cả ba bắt đầu từ pretrained revision gốc. Script lưu từng lượt vào thư mục riêng và kiểm tra kết quả từ xác suất. Thư mục kết quả/checkpoint đã tồn tại sẽ bị từ chối để tránh ghi đè. Thời gian baseline cũ khoảng 51 phút/lượt; thời gian thực tế sweep chưa đo.

In [ ]:
import py_vncorenlp
segmenter_dir = (ROOT/'data/vncorenlp').resolve()
segmenter_dir.mkdir(parents=True,exist_ok=True)
py_vncorenlp.download_model(save_dir=str(segmenter_dir))
subprocess.run([sys.executable,'scripts/tune_phobert.py','run',
                '--vncorenlp-dir',str(segmenter_dir)],check=True)

## Xem kết quả và tạo lại report/slide
Các insight về learning rate cần dựa trên số liệu mới. Bảng ca lỗi tự sinh còn cần nhóm đọc thủ công; không đưa comment nguồn vào báo cáo công khai.

In [ ]:
import pandas as pd
from IPython.display import display, Image
subprocess.run([sys.executable,'scripts/build_report.py'],check=True)
display(pd.read_csv('reports/phobert_tuning/comparison.csv'))
display(pd.read_csv('reports/phobert_tuning/styles_comparison.csv'))
display(Image(filename='reports/phobert_tuning/learning_curves.png'))
display(pd.read_csv('reports/phobert_tuning/neural_error_review.csv').head())

In [ ]:
with zipfile.ZipFile('/content/vigoemotions_tuning_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ('reports','splits'):
        for p in Path(folder).rglob('*'):
            if p.is_file():
                z.write(p,str(p))
files.download('/content/vigoemotions_tuning_results.zip')

## Tải checkpoint được chọn để giữ cho Final
Lưu file này riêng tư trước khi runtime hết hạn. Checkpoint có kích thước lớn và không nằm trong ZIP kết quả. Chưa đánh giá test; Final sẽ dùng checkpoint đã chốt để so sánh ba cách chọn threshold.

In [ ]:
study=json.loads(Path('reports/phobert_tuning/study.json').read_text())
selected=study['selected_run']
run=json.loads(Path(f'reports/phobert_tuning/{selected}/phobert_run.json').read_text())
checkpoint=Path(run['checkpoint_path'])
assert checkpoint.is_file()
files.download(str(checkpoint))